# 📊 Explorative Datenanalyse (EDA) - Controlling Edition

**Zweck:** Automatisierte, explorative Analyse von Excel- oder CSV-Dateien mit Fokus auf managementtaugliche Insights.

**Erstellt:** 2026-01-21  
**Version:** 1.0

---

## 🎯 Was macht dieses Notebook?

- ✅ Automatische Erkennung von Measures (Kennzahlen) und Dimensionen (Kategorien)
- ✅ Umfassende Datenqualitätsprüfung
- ✅ Management-taugliche Visualisierungen (KPIs, Top-N, Pareto, Outlier)
- ✅ Treiberanalyse und Konzentrationsanalyse
- ✅ Executive Summary mit konkreten Next Steps

---

## A) Setup & Imports

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime
import re
from typing import List, Dict, Tuple, Optional

# Visualization Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Imports erfolgreich geladen")
print(f"📦 Pandas Version: {pd.__version__}")
print(f"📦 NumPy Version: {np.__version__}")

## B) Konfiguration & Parameter

**HIER EINGEBEN:** Passen Sie die folgenden Parameter an Ihre Datei an.

In [ ]:
# ========================================
# KONFIGURATION - BITTE ANPASSEN
# ========================================

# --- INPUT PARAMETER ---
FILE_PATH = "<PFAD_ZUR_DATEI>"  # z.B. "data/sales_2025.xlsx" oder "data/transactions.csv"
SHEET_NAME = "<SHEET_ODER_LEER>"  # Bei Excel: Sheet-Name oder None/"" für erstes Sheet
CSV_DELIMITER = "<DELIMITER_ODER_LEER>"  # Bei CSV: "," oder ";" oder None für Auto-Detection

# --- OPTIONALE SPALTEN-OVERRIDE (wenn automatische Erkennung nicht ausreicht) ---
FORCED_MEASURES = []  # z.B. ["Umsatz", "Menge", "Kosten"]
FORCED_DIMENSIONS = []  # z.B. ["Kunde", "Produktgruppe", "Land"]

# --- ANALYSE-PARAMETER ---
TOP_N = 10  # Anzahl Top-Dimensionen für Analysen
OUTLIER_IQR_MULTIPLIER = 1.5  # Ausreißer-Schwelle (Standard: 1.5 * IQR)
PARETO_THRESHOLD = 0.8  # 80% für Pareto-Analyse
MIN_CORRELATION = 0.3  # Minimale Korrelation für Heatmap
MAX_UNIQUE_FOR_DIMENSION = 500  # Max. Unique Values für Dimension-Klassifikation

# --- ENCODING (für CSV) ---
ENCODINGS_TO_TRY = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']

print("✅ Konfiguration geladen")
print(f"📁 Datei: {FILE_PATH}")

## C) Hilfsfunktionen

In [ ]:
def load_data_smart(file_path: str, sheet_name: Optional[str] = None, 
                   delimiter: Optional[str] = None) -> pd.DataFrame:
    """
    Intelligentes Laden von Excel oder CSV mit Fehlerbehandlung.
    """
    file_path_obj = Path(file_path)
    
    # Existenzprüfung
    if not file_path_obj.exists():
        raise FileNotFoundError(f"❌ Datei nicht gefunden: {file_path}")
    
    file_ext = file_path_obj.suffix.lower()
    
    # Excel
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        try:
            if sheet_name and sheet_name != "<SHEET_ODER_LEER>" and sheet_name.strip():
                df = pd.read_excel(file_path, sheet_name=sheet_name)
                print(f"✅ Excel geladen: Sheet '{sheet_name}'")
            else:
                df = pd.read_excel(file_path, sheet_name=0)
                print(f"✅ Excel geladen: Erstes Sheet verwendet")
            return df
        except Exception as e:
            raise ValueError(f"❌ Fehler beim Laden der Excel-Datei: {e}")
    
    # CSV
    elif file_ext == '.csv':
        # Delimiter Detection
        if delimiter and delimiter != "<DELIMITER_ODER_LEER>" and delimiter.strip():
            delim = delimiter
        else:
            # Auto-detect
            with open(file_path, 'r', encoding='utf-8') as f:
                first_line = f.readline()
                if ';' in first_line:
                    delim = ';'
                elif '\t' in first_line:
                    delim = '\t'
                else:
                    delim = ','
        
        # Encoding Detection
        for encoding in ENCODINGS_TO_TRY:
            try:
                df = pd.read_csv(file_path, sep=delim, encoding=encoding)
                print(f"✅ CSV geladen: Delimiter='{delim}', Encoding='{encoding}'")
                return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
        
        raise ValueError(f"❌ Keine passende Encoding gefunden. Versucht: {ENCODINGS_TO_TRY}")
    
    else:
        raise ValueError(f"❌ Nicht unterstütztes Dateiformat: {file_ext}. Nur .xlsx, .xls, .csv erlaubt.")


def detect_column_types(df: pd.DataFrame, forced_measures: List[str] = None, 
                       forced_dimensions: List[str] = None) -> Dict[str, List[str]]:
    """
    Klassifiziert Spalten automatisch in Measures, Dimensions und Dates.
    """
    forced_measures = forced_measures or []
    forced_dimensions = forced_dimensions or []
    
    measures = []
    dimensions = []
    date_columns = []
    
    # Heuristiken für Measure-Namen
    measure_patterns = [
        r'(?i)(umsatz|revenue|sales|amount|betrag|wert|value|kosten|cost|price|preis)',
        r'(?i)(menge|quantity|qty|anzahl|count|volumen|volume)',
        r'(?i)(eur|usd|euro|dollar|chf)',
        r'(?i)(margin|marge|gewinn|profit|ebitda|ebit)',
        r'(?i)(kpi|kennzahl|metric)',
    ]
    
    for col in df.columns:
        # Forced Override
        if col in forced_measures:
            measures.append(col)
            continue
        if col in forced_dimensions:
            dimensions.append(col)
            continue
        
        # Datum-Erkennung
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            date_columns.append(col)
            continue
        
        # Versuche datetime Parsing
        if df[col].dtype == 'object':
            try:
                pd.to_datetime(df[col].dropna().head(100), errors='coerce')
                if pd.to_datetime(df[col].dropna(), errors='coerce').notna().sum() / len(df[col].dropna()) > 0.8:
                    date_columns.append(col)
                    continue
            except:
                pass
        
        # Numerische Spalten
        if pd.api.types.is_numeric_dtype(df[col]):
            # Prüfe, ob es eine ID ist (viele Unique Values, ganzzahlig)
            nunique = df[col].nunique()
            is_integer = df[col].dropna().apply(lambda x: float(x).is_integer()).all()
            
            # Heuristik: IDs haben >80% Unique Values
            if nunique / len(df) > 0.8 and is_integer:
                dimensions.append(col)
            # Heuristik: Wenige Unique Values = Kategorie
            elif nunique < MAX_UNIQUE_FOR_DIMENSION:
                dimensions.append(col)
            else:
                # Check Name Pattern
                is_measure_name = any(re.search(pattern, col) for pattern in measure_patterns)
                if is_measure_name:
                    measures.append(col)
                else:
                    measures.append(col)  # Default: numerisch = Measure
        
        # Text/Objekt-Spalten
        else:
            nunique = df[col].nunique()
            if nunique < MAX_UNIQUE_FOR_DIMENSION:
                dimensions.append(col)
            else:
                # Zu viele Unique Values (z.B. Freitext) -> als Dimension, aber mit Warnung
                dimensions.append(col)
    
    return {
        'measures': measures,
        'dimensions': dimensions,
        'dates': date_columns
    }


def format_number(num: float) -> str:
    """Formatiert Zahlen für Management-Reports."""
    if pd.isna(num):
        return "N/A"
    if abs(num) >= 1_000_000:
        return f"{num/1_000_000:.1f}M"
    elif abs(num) >= 1_000:
        return f"{num/1_000:.1f}K"
    else:
        return f"{num:.2f}"


print("✅ Hilfsfunktionen definiert")

## D) Datei laden

In [ ]:
# Datei laden mit Fehlerbehandlung
try:
    df = load_data_smart(
        file_path=FILE_PATH,
        sheet_name=SHEET_NAME if SHEET_NAME != "<SHEET_ODER_LEER>" else None,
        delimiter=CSV_DELIMITER if CSV_DELIMITER != "<DELIMITER_ODER_LEER>" else None
    )
    
    # Validierung
    if df.empty:
        raise ValueError("❌ Datei ist leer!")
    
    print(f"\n✅ Daten erfolgreich geladen!")
    print(f"📊 Shape: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
    print(f"💾 Speicher: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
except Exception as e:
    print(f"\n❌ FEHLER beim Laden: {e}")
    print("\n💡 Bitte prüfen Sie:")
    print("   - Ist der Dateipfad korrekt?")
    print("   - Existiert die Datei?")
    print("   - Bei Excel: Ist der Sheet-Name korrekt?")
    print("   - Bei CSV: Ist das Trennzeichen korrekt?")
    raise

## E) Erstprofil der Daten

In [ ]:
print("=" * 80)
print("ERST-PROFIL DER DATEN")
print("=" * 80)

print(f"\n📌 Anzahl Zeilen: {df.shape[0]:,}")
print(f"📌 Anzahl Spalten: {df.shape[1]}")
print(f"📌 Speicherverbrauch: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "─" * 80)
print("ERSTE 5 ZEILEN:")
print("─" * 80)
display(df.head())

print("\n" + "─" * 80)
print("DATENTYPEN:")
print("─" * 80)
dtype_summary = df.dtypes.value_counts()
print(dtype_summary)

print("\n" + "─" * 80)
print("SPALTEN-ÜBERSICHT:")
print("─" * 80)
col_info = pd.DataFrame({
    'Spalte': df.columns,
    'Typ': df.dtypes.values,
    'Non-Null': df.notna().sum().values,
    'Null %': (df.isna().sum() / len(df) * 100).values,
    'Unique': [df[col].nunique() for col in df.columns]
})
display(col_info)

## F) Spaltenklassifikation: Measures vs. Dimensionen vs. Datum

In [ ]:
# Automatische Klassifikation
column_types = detect_column_types(
    df, 
    forced_measures=FORCED_MEASURES if FORCED_MEASURES else None,
    forced_dimensions=FORCED_DIMENSIONS if FORCED_DIMENSIONS else None
)

measures = column_types['measures']
dimensions = column_types['dimensions']
date_columns = column_types['dates']

print("=" * 80)
print("SPALTEN-KLASSIFIKATION")
print("=" * 80)

print(f"\n📊 MEASURES (Kennzahlen): {len(measures)}")
for m in measures:
    print(f"   - {m}")

print(f"\n📁 DIMENSIONEN (Kategorien): {len(dimensions)}")
for d in dimensions:
    nunique = df[d].nunique()
    print(f"   - {d} ({nunique:,} unique)")

print(f"\n📅 DATUM-SPALTEN: {len(date_columns)}")
for dt in date_columns:
    print(f"   - {dt}")

if not measures:
    print("\n⚠️  WARNUNG: Keine Measures erkannt! Bitte FORCED_MEASURES setzen.")
if not dimensions:
    print("\n⚠️  WARNUNG: Keine Dimensionen erkannt! Bitte FORCED_DIMENSIONS setzen.")

## G) Datenqualitäts-Check (Systematisch)

In [ ]:
print("=" * 80)
print("DATENQUALITÄTS-CHECK")
print("=" * 80)

quality_issues = []

# 1. Missing Values
print("\n📍 1) MISSING VALUES")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Spalte': missing.index,
    'Missing': missing.values,
    'Missing %': missing_pct.values
}).query('Missing > 0').sort_values('Missing', ascending=False)

if len(missing_df) > 0:
    display(missing_df)
    for _, row in missing_df.iterrows():
        if row['Missing %'] > 50:
            quality_issues.append(f"❌ {row['Spalte']}: {row['Missing %']:.1f}% fehlend (kritisch!)")
        elif row['Missing %'] > 10:
            quality_issues.append(f"⚠️  {row['Spalte']}: {row['Missing %']:.1f}% fehlend")
else:
    print("   ✅ Keine Missing Values gefunden")

# 2. Duplikate
print("\n📍 2) DUPLIKATE")
duplicates = df.duplicated().sum()
dup_pct = (duplicates / len(df) * 100).round(2)
print(f"   Duplikate: {duplicates:,} ({dup_pct}%)")
if duplicates > 0:
    quality_issues.append(f"⚠️  {duplicates:,} duplizierte Zeilen gefunden ({dup_pct}%)")
else:
    print("   ✅ Keine Duplikate gefunden")

# 3. Null/Negative Werte in Measures
print("\n📍 3) NULL & NEGATIVE WERTE (Measures)")
for measure in measures:
    zeros = (df[measure] == 0).sum()
    negatives = (df[measure] < 0).sum()
    zeros_pct = (zeros / len(df) * 100).round(2)
    neg_pct = (negatives / len(df) * 100).round(2)
    
    print(f"   {measure}:")
    print(f"      Nullen: {zeros:,} ({zeros_pct}%)")
    print(f"      Negative: {negatives:,} ({neg_pct}%)")
    
    if zeros_pct > 30:
        quality_issues.append(f"⚠️  {measure}: {zeros_pct}% Null-Werte")
    if negatives > 0 and 'kosten' not in measure.lower() and 'margin' not in measure.lower():
        quality_issues.append(f"⚠️  {measure}: {negatives:,} negative Werte (prüfen!)")

# 4. Extreme Werte / Ausreißer
print("\n📍 4) AUSREISSER-CHECK (IQR-Methode)")
for measure in measures:
    q1 = df[measure].quantile(0.25)
    q3 = df[measure].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - OUTLIER_IQR_MULTIPLIER * iqr
    upper_bound = q3 + OUTLIER_IQR_MULTIPLIER * iqr
    
    outliers = df[(df[measure] < lower_bound) | (df[measure] > upper_bound)][measure]
    outlier_pct = (len(outliers) / len(df) * 100).round(2)
    
    print(f"   {measure}: {len(outliers):,} Ausreißer ({outlier_pct}%)")
    if outlier_pct > 5:
        quality_issues.append(f"⚠️  {measure}: {outlier_pct}% Ausreißer (prüfen!)")

# 5. Zusammenfassung
print("\n" + "=" * 80)
print("QUALITÄTS-ISSUES ZUSAMMENFASSUNG")
print("=" * 80)
if quality_issues:
    for issue in quality_issues:
        print(f"   {issue}")
else:
    print("   ✅ Keine kritischen Qualitätsprobleme gefunden!")

## H) Measures Deep Dive: Verteilungen, Ausreißer, Statistiken

In [ ]:
if not measures:
    print("⚠️  Keine Measures vorhanden - Abschnitt wird übersprungen")
else:
    print("=" * 80)
    print("MEASURES DEEP DIVE")
    print("=" * 80)
    
    # Deskriptive Statistik
    print("\n📊 DESKRIPTIVE STATISTIK")
    stats = df[measures].describe().T
    stats['sum'] = df[measures].sum()
    stats['missing'] = df[measures].isna().sum()
    stats['zeros'] = (df[measures] == 0).sum()
    display(stats)
    
    # Verteilungsplots
    print("\n📊 VERTEILUNGEN (Histogramm + Boxplot)")
    n_measures = len(measures)
    fig, axes = plt.subplots(n_measures, 2, figsize=(14, 4 * n_measures))
    
    if n_measures == 1:
        axes = axes.reshape(1, -1)
    
    for idx, measure in enumerate(measures):
        # Histogram
        ax1 = axes[idx, 0]
        df[measure].dropna().hist(bins=50, ax=ax1, edgecolor='black')
        ax1.set_title(f'Verteilung: {measure}')
        ax1.set_xlabel(measure)
        ax1.set_ylabel('Häufigkeit')
        ax1.axvline(df[measure].mean(), color='red', linestyle='--', label=f'Mean: {format_number(df[measure].mean())}')
        ax1.axvline(df[measure].median(), color='green', linestyle='--', label=f'Median: {format_number(df[measure].median())}')
        ax1.legend()
        
        # Boxplot
        ax2 = axes[idx, 1]
        df.boxplot(column=measure, ax=ax2, vert=False)
        ax2.set_title(f'Boxplot: {measure}')
        ax2.set_xlabel(measure)
    
    plt.tight_layout()
    plt.show()

## I) Dimension Impact: Measures nach Dimensionen (Treiberanalyse)

In [ ]:
if not measures or not dimensions:
    print("⚠️  Keine Measures oder Dimensionen vorhanden - Abschnitt wird übersprungen")
else:
    print("=" * 80)
    print("DIMENSION IMPACT ANALYSE")
    print("=" * 80)
    
    # Wähle wichtigste Measure (höchste Summe)
    primary_measure = df[measures].sum().idxmax()
    print(f"\n🎯 Primäre Measure für Analyse: {primary_measure}")
    
    # Analyse für jede Dimension
    for dim in dimensions:
        print(f"\n📊 Dimension: {dim}")
        
        # Nur Dimensionen mit <1000 Unique Values detailliert analysieren
        nunique = df[dim].nunique()
        if nunique > 1000:
            print(f"   ⚠️  Zu viele Unique Values ({nunique:,}) - überspringe detaillierte Analyse")
            continue
        
        # Top-N nach Measure
        top_n_data = df.groupby(dim)[primary_measure].sum().sort_values(ascending=False).head(TOP_N)
        
        if len(top_n_data) == 0:
            print(f"   ⚠️  Keine Daten für {dim}")
            continue
        
        total = df[primary_measure].sum()
        top_n_sum = top_n_data.sum()
        top_n_pct = (top_n_sum / total * 100) if total > 0 else 0
        
        print(f"   Top-{TOP_N} Anteil an {primary_measure}: {top_n_pct:.1f}%")
        
        # Konzentration (Top-3)
        top3_pct = (top_n_data.head(3).sum() / total * 100) if total > 0 else 0
        print(f"   Top-3 Konzentration: {top3_pct:.1f}%")
        
        # Visualisierung nur für Top-Dimensionen
        if nunique <= 50:  # Nur für überschaubare Dimensionen
            fig, ax = plt.subplots(figsize=(12, 6))
            top_n_data.plot(kind='barh', ax=ax, color='steelblue')
            ax.set_title(f'Top-{TOP_N}: {primary_measure} nach {dim}')
            ax.set_xlabel(primary_measure)
            ax.set_ylabel(dim)
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()

## J) Visual Story: Management-taugliche Dashboards

### J.1) KPI-Übersicht (Kennzahlenkarten)

In [ ]:
if not measures:
    print("⚠️  Keine Measures vorhanden")
else:
    print("=" * 80)
    print("KPI-ÜBERSICHT")
    print("=" * 80)
    
    kpi_data = []
    for measure in measures:
        kpi_data.append({
            'KPI': measure,
            'Summe': format_number(df[measure].sum()),
            'Durchschnitt': format_number(df[measure].mean()),
            'Median': format_number(df[measure].median()),
            'Min': format_number(df[measure].min()),
            'Max': format_number(df[measure].max()),
            'Std. Abw.': format_number(df[measure].std())
        })
    
    kpi_df = pd.DataFrame(kpi_data)
    display(kpi_df)
    
    # Visualisierung als Balkendiagramm
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(measures))
    totals = [df[m].sum() for m in measures]
    
    bars = ax.bar(x, totals, color='steelblue', edgecolor='black')
    ax.set_xlabel('Measures')
    ax.set_ylabel('Summe')
    ax.set_title('KPI-Übersicht: Summen aller Measures')
    ax.set_xticks(x)
    ax.set_xticklabels(measures, rotation=45, ha='right')
    
    # Werte auf Balken
    for bar, val in zip(bars, totals):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                format_number(val),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### J.2) Pareto-Analyse (80/20-Regel)

In [ ]:
if not measures or not dimensions:
    print("⚠️  Keine Measures oder Dimensionen vorhanden")
else:
    print("=" * 80)
    print("PARETO-ANALYSE (80/20-Regel)")
    print("=" * 80)
    
    # Wähle primäre Measure und erste geeignete Dimension
    primary_measure = df[measures].sum().idxmax()
    
    # Finde erste Dimension mit <500 Unique Values
    suitable_dim = None
    for dim in dimensions:
        if df[dim].nunique() < 500:
            suitable_dim = dim
            break
    
    if not suitable_dim:
        print(f"⚠️  Keine geeignete Dimension für Pareto gefunden (alle >500 unique)")
    else:
        # Pareto-Berechnung
        pareto_data = df.groupby(suitable_dim)[primary_measure].sum().sort_values(ascending=False)
        pareto_data_cumsum = pareto_data.cumsum()
        pareto_data_cum_pct = (pareto_data_cumsum / pareto_data.sum() * 100)
        
        # Wie viele Items für 80%?
        items_for_80 = (pareto_data_cum_pct <= PARETO_THRESHOLD * 100).sum()
        pct_items_for_80 = (items_for_80 / len(pareto_data) * 100)
        
        print(f"\n🎯 Pareto-Analyse: {primary_measure} nach {suitable_dim}")
        print(f"   {items_for_80} von {len(pareto_data)} Items ({pct_items_for_80:.1f}%) machen {PARETO_THRESHOLD*100:.0f}% des {primary_measure} aus")
        
        # Visualisierung
        fig, ax1 = plt.subplots(figsize=(14, 7))
        
        # Balken
        ax1.bar(range(len(pareto_data)), pareto_data.values, color='steelblue', label=primary_measure)
        ax1.set_xlabel(suitable_dim)
        ax1.set_ylabel(primary_measure, color='steelblue')
        ax1.tick_params(axis='y', labelcolor='steelblue')
        
        # Kumulative Linie
        ax2 = ax1.twinx()
        ax2.plot(range(len(pareto_data_cum_pct)), pareto_data_cum_pct.values, 
                color='red', marker='o', linewidth=2, label='Kumuliert %')
        ax2.axhline(y=PARETO_THRESHOLD*100, color='green', linestyle='--', 
                   linewidth=2, label=f'{PARETO_THRESHOLD*100:.0f}%-Linie')
        ax2.set_ylabel('Kumuliert %', color='red')
        ax2.tick_params(axis='y', labelcolor='red')
        ax2.set_ylim([0, 105])
        
        plt.title(f'Pareto-Chart: {primary_measure} nach {suitable_dim}\n({items_for_80} Items = {PARETO_THRESHOLD*100:.0f}%)')
        ax1.legend(loc='upper left')
        ax2.legend(loc='upper right')
        
        # Nur Top-50 anzeigen (bei zu vielen Items)
        if len(pareto_data) > 50:
            ax1.set_xlim([0, 50])
            print(f"   ℹ️  Chart zeigt nur Top-50 (Gesamt: {len(pareto_data)})")
        
        plt.tight_layout()
        plt.show()

### J.3) Ausreißer-Visualisierung

In [ ]:
if not measures:
    print("⚠️  Keine Measures vorhanden")
else:
    print("=" * 80)
    print("AUSREISSER-VISUALISIERUNG")
    print("=" * 80)
    
    fig, axes = plt.subplots(1, len(measures), figsize=(6 * len(measures), 6))
    
    if len(measures) == 1:
        axes = [axes]
    
    for idx, measure in enumerate(measures):
        ax = axes[idx]
        
        # Boxplot mit Violin overlay
        parts = ax.violinplot([df[measure].dropna()], positions=[0], widths=0.7,
                             showmeans=True, showmedians=True)
        
        ax.set_title(f'Ausreißer-View: {measure}')
        ax.set_ylabel(measure)
        ax.set_xticks([0])
        ax.set_xticklabels([measure])
        
        # Statistiken
        q1 = df[measure].quantile(0.25)
        q3 = df[measure].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - OUTLIER_IQR_MULTIPLIER * iqr
        upper = q3 + OUTLIER_IQR_MULTIPLIER * iqr
        outliers = df[(df[measure] < lower) | (df[measure] > upper)][measure]
        
        ax.text(0.5, 0.95, f'Ausreißer: {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)',
               transform=ax.transAxes, ha='center', va='top',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

### J.4) Korrelations-Heatmap (nur bei >2 Measures)

In [ ]:
if len(measures) < 2:
    print("⚠️  Weniger als 2 Measures - Korrelation nicht sinnvoll")
else:
    print("=" * 80)
    print("KORRELATIONS-HEATMAP")
    print("=" * 80)
    
    # Berechne Korrelation
    corr_matrix = df[measures].corr()
    
    # Visualisierung
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Heatmap
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
                vmin=-1, vmax=1, ax=ax)
    
    ax.set_title('Korrelations-Matrix: Measures')
    plt.tight_layout()
    plt.show()
    
    # Starke Korrelationen hervorheben
    print(f"\n🔍 Starke Korrelationen (|r| > {MIN_CORRELATION}):")
    strong_corr = []
    for i in range(len(corr_matrix)):
        for j in range(i+1, len(corr_matrix)):
            if abs(corr_matrix.iloc[i, j]) > MIN_CORRELATION:
                strong_corr.append({
                    'Measure 1': corr_matrix.index[i],
                    'Measure 2': corr_matrix.columns[j],
                    'Korrelation': corr_matrix.iloc[i, j]
                })
    
    if strong_corr:
        strong_corr_df = pd.DataFrame(strong_corr).sort_values('Korrelation', 
                                                                ascending=False, key=abs)
        display(strong_corr_df)
    else:
        print(f"   Keine starken Korrelationen (|r| > {MIN_CORRELATION}) gefunden")

## K) Executive Summary + Next Steps

In [ ]:
# Sammle Key Insights automatisch
insights = []
next_steps = []

# Insight 1: Datenvolumen
insights.append(f"📊 **Datenumfang:** {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten, {len(measures)} Measures, {len(dimensions)} Dimensionen")

# Insight 2: Datenqualität
missing_critical = missing_df[missing_df['Missing %'] > 10] if len(missing_df) > 0 else pd.DataFrame()
if len(missing_critical) > 0:
    insights.append(f"⚠️  **Datenqualität:** {len(missing_critical)} Spalte(n) mit >10% Missing Values")
    next_steps.append("Prüfen Sie die Missing Values: Sind diese systematisch oder zufällig? Imputation oder Ausschluss?")
else:
    insights.append("✅ **Datenqualität:** Keine kritischen Missing Values (<10%)")

# Insight 3: Duplikate
if duplicates > 0:
    insights.append(f"⚠️  **Duplikate:** {duplicates:,} duplizierte Zeilen ({dup_pct}%)")
    next_steps.append("Duplikate prüfen: Echte Duplikate oder legitime Wiederholungen?")

# Insight 4: Top Measure
if measures:
    primary_measure = df[measures].sum().idxmax()
    primary_sum = df[primary_measure].sum()
    insights.append(f"💰 **Wichtigste Kennzahl:** {primary_measure} (Summe: {format_number(primary_sum)})")

# Insight 5: Pareto/Konzentration
if suitable_dim:
    insights.append(f"📈 **Konzentration (Pareto):** {items_for_80} von {len(pareto_data)} {suitable_dim} ({pct_items_for_80:.1f}%) machen {PARETO_THRESHOLD*100:.0f}% des {primary_measure} aus")
    if pct_items_for_80 < 30:
        next_steps.append(f"Hohe Konzentration bei {suitable_dim}: Risiko-Check für Abhängigkeiten von wenigen Top-Kunden/Produkten")

# Insight 6: Ausreißer
if measures:
    outlier_summary = []
    for measure in measures:
        q1 = df[measure].quantile(0.25)
        q3 = df[measure].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - OUTLIER_IQR_MULTIPLIER * iqr
        upper = q3 + OUTLIER_IQR_MULTIPLIER * iqr
        outliers = df[(df[measure] < lower) | (df[measure] > upper)][measure]
        outlier_pct = len(outliers) / len(df) * 100
        if outlier_pct > 5:
            outlier_summary.append(f"{measure} ({outlier_pct:.1f}%)")
    
    if outlier_summary:
        insights.append(f"🔍 **Ausreißer:** Erhöhte Ausreißer-Quote bei: {', '.join(outlier_summary)}")
        next_steps.append("Ausreißer analysieren: Datenfehler, Sonderfälle oder echte Extremwerte?")

# Insight 7: Korrelationen
if len(measures) >= 2 and strong_corr:
    top_corr = strong_corr_df.iloc[0]
    insights.append(f"🔗 **Stärkste Korrelation:** {top_corr['Measure 1']} ↔ {top_corr['Measure 2']} (r={top_corr['Korrelation']:.2f})")
    if abs(top_corr['Korrelation']) > 0.8:
        next_steps.append(f"Sehr starke Korrelation zwischen {top_corr['Measure 1']} und {top_corr['Measure 2']}: Kausaler Zusammenhang oder Multikollinearität?")

# Standard Next Steps
next_steps.append("Definieren Sie Business-Rules für Ausreißer und Missing Values")
next_steps.append("Vertiefen Sie die Analyse der Top-Treiber (Kundensegmente, Produkte, Regionen)")
if date_columns:
    next_steps.append(f"Zeitreihen-Analyse für {', '.join(date_columns[:2])} durchführen")

# Print Executive Summary
print("=" * 80)
print("📋 EXECUTIVE SUMMARY")
print("=" * 80)
print("\n**🎯 KEY INSIGHTS:**\n")
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n" + "─" * 80)
print("\n**🚀 NEXT STEPS / HANDLUNGSEMPFEHLUNGEN:**\n")
for i, step in enumerate(next_steps, 1):
    print(f"{i}. {step}")

print("\n" + "=" * 80)

## L) Appendix: Reproduzierbarkeit & Parameter

In [ ]:
print("=" * 80)
print("📋 APPENDIX: PARAMETER & REPRODUZIERBARKEIT")
print("=" * 80)

print("\n📌 VERWENDETE PARAMETER:")
print(f"   - Datei: {FILE_PATH}")
print(f"   - Sheet: {SHEET_NAME if SHEET_NAME != '<SHEET_ODER_LEER>' else 'Auto (erstes Sheet)'}")
print(f"   - CSV Delimiter: {CSV_DELIMITER if CSV_DELIMITER != '<DELIMITER_ODER_LEER>' else 'Auto-Detection'}")
print(f"   - Top-N: {TOP_N}")
print(f"   - Outlier IQR Multiplier: {OUTLIER_IQR_MULTIPLIER}")
print(f"   - Pareto Threshold: {PARETO_THRESHOLD}")
print(f"   - Min Correlation: {MIN_CORRELATION}")
print(f"   - Max Unique für Dimension: {MAX_UNIQUE_FOR_DIMENSION}")

print("\n📌 SPALTEN-KLASSIFIKATION:")
print(f"   - Measures: {measures}")
print(f"   - Dimensions: {dimensions}")
print(f"   - Date Columns: {date_columns}")
print(f"   - Forced Measures: {FORCED_MEASURES if FORCED_MEASURES else 'Keine'}")
print(f"   - Forced Dimensions: {FORCED_DIMENSIONS if FORCED_DIMENSIONS else 'Keine'}")

print("\n📌 SYSTEM-INFO:")
print(f"   - Analyse-Datum: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   - Pandas Version: {pd.__version__}")
print(f"   - NumPy Version: {np.__version__}")

print("\n✅ NOTEBOOK ABGESCHLOSSEN")
print("=" * 80)

---

## 🎓 Nutzungshinweise

**So verwenden Sie dieses Notebook:**

1. **Parameter einstellen** (Zelle B): Passen Sie `FILE_PATH`, `SHEET_NAME`, etc. an
2. **Run All**: Führen Sie alle Zellen aus (Kernel → Restart & Run All)
3. **Review Executive Summary**: Am Ende finden Sie die wichtigsten Insights
4. **Export**: Speichern Sie Visualisierungen oder exportieren Sie das Notebook als PDF/HTML

**Troubleshooting:**
- Bei Encoding-Problemen: Probieren Sie andere Encodings in `ENCODINGS_TO_TRY`
- Bei falscher Spalten-Klassifikation: Nutzen Sie `FORCED_MEASURES` / `FORCED_DIMENSIONS`
- Bei zu vielen Dimensionen: Erhöhen Sie `MAX_UNIQUE_FOR_DIMENSION`

**Datenschutz:**
- Dieses Notebook zeigt nur aggregierte Daten
- Keine Einzeldatensätze werden exportiert
- Bei sensiblen Daten: Anonymisieren Sie vor dem Teilen

---

*Erstellt mit Claude Code - EDA Notebook Creator v1.0*